In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import cv2
import sys
import os


home = Path.home()
#sys.path.append(os.path.join(home,"analysis/multiplex/raw_counter/src"))
#sys.path.append(os.path.join(Path.cwd(), 'src'))
if "ivana" in str(home):
    str_path =  "Dropbox\Ivana Ducrey\Doctorado- Drosophila melanogaster\Berlin Lisa\analysis\behavior\raw_counter\src"
    path2add = str(home) + str_path
else:
     path2add = str(home) + "analysis/multiplex/raw_counter/src"
print(str_path)
sys.path.append(path2add)
from counter_functions import *

home_dir = os.path.join(home, "data/multiplex")
#home_dir = Path.home() / "data" / "multiplex"
print(home_dir)






The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
aw_counter\srcDropbox\Ivana Ducrey\Doctorado- Drosophila melanogaster\Berlin Lisanalysiehavior


ModuleNotFoundError: No module named 'counter_functions'

In [20]:


date = '10_29_2026'
movies2check = [
               0, 
                1,
                ]

loc_training_oddors = ['down', 'up'] # up = MCH, down = Octanol
#movie_locs = [f"{home_dir}/{date}/{date}_movie_{num}.mp4" for num in movies2check]
paired_exp = True
PIs =[]
PIs_final = []

do_not_show = False



In [ ]:
for i, movie_number in enumerate(movies2check):
    # create directory to store frames etc..
    out_dir = f"{home_dir}/{date}/{date}_movie_{movie_number}"
    os.makedirs(out_dir, exist_ok=True)
    movie_loc = f"{home_dir}/{date}/{date}_movie_{movie_number}.mp4"
    frames = xtract_frames(movie_loc)
    # selected frames/ 
    last = np.median(np.stack(frames[-10:]), axis=0).astype(np.uint8)   # ~0.7 s
    background_frame = background(frames)
    background_aligned = align_background(background_frame, last)

    #localize flies:
    diff, mask_flies, overlay = loc_flies(background_aligned, last, threshold=15)
    
    
    # check or draw ROIs:
    roi_file = f"{out_dir}/{date}_rois_{movie_number}.json"
    
    rois, cid = load_or_draw_rois(overlay, roi_file, update_date_rois=True, do_not_show = do_not_show)
    
    
    #count flies
    overlay, counts, pi_maze = count_flies(mask_flies, overlay, rois, loc_training_oddors[i])
    PIs.append(pi_maze)
    
    # store images
    cv2.imwrite(f"{out_dir}/{date}_last_{movie_number}.png", last)
    cv2.imwrite(f"{out_dir}/{date}_background_{movie_number}.png", background_aligned)
    cv2.imwrite(f"{out_dir}/{date}_diff_{movie_number}.png", cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX))
    cv2.imwrite(f"{out_dir}/{date}_mask_flies_{movie_number}.png", mask_flies)
    cv2.imwrite(f"{out_dir}/{date}_overlay_{movie_number}.png", overlay)
    print("*******************  MOVIE DONE  ****************")
if paired_exp:
    for maze, val in enumerate(PIs[0]):
        if ~np.isnan(PIs[0][maze]):
            print((PIs[0][maze] + PIs[1][maze])/2)

            

    
